# OVD-Diagnose — Domain 2: Medical (VinDr chest X-ray)

Second specialized domain. VinDr = 14 finding classes with boxes. Expected contrast vs
aerial: subtle low-contrast findings + weak CLIP semantics → **L / C-bound**, not S-bound.

**Data:** Kaggle → Add Data → `awsaf49/vinbigdata-512-image-dataset` (512px PNGs + train.csv).

In [ ]:
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| GPUs', torch.cuda.device_count())

In [ ]:
REPO = 'https://github.com/ShMazumder/YOLOBERT.git'
import os
if not os.path.isdir('/kaggle/working/YOLOBERT'):
    !git clone $REPO /kaggle/working/YOLOBERT
%cd /kaggle/working/YOLOBERT
!git pull -q || true
!pip install -q ultralytics transformers pycocotools pyyaml pillow

In [ ]:
CSV  = '/kaggle/input/datasets/awsaf49/vinbigdata-512-image-dataset/vinbigdata/train.csv'
IMGS = '/kaggle/input/datasets/awsaf49/vinbigdata-512-image-dataset/vinbigdata/train'
OUT  = 'data/medical/annotations/instances_val.json'
import os
print('csv exists:', os.path.exists(CSV), '| imgs exists:', os.path.isdir(IMGS))
print('csv header:', open(CSV).readline().strip())

In [ ]:
# Convert VinDr train.csv -> COCO json (14 classes, boxes auto-scaled to PNG size)
!python tools/vindr_to_coco.py --csv "{CSV}" --imgs "{IMGS}" --out "{OUT}" --img_ext .png

In [ ]:
# Box-alignment check (should print OK, max coord <= 512)
from pycocotools.coco import COCO
c = COCO(OUT); bb = [a['bbox'] for a in c.loadAnns(c.getAnnIds())[:8]]
mx = max(v for b in bb for v in b)
print('max box coord:', round(mx), '-> OK (scaled)' if mx <= 600 else '-> UNSCALED')

In [ ]:
# Run all models
LIMIT = 0   # 0 = full (4394 imgs); 200 for a quick pass
limit_arg = f'--limit {LIMIT}' if LIMIT else ''
!python tools/run_all.py \
  --ann  data/medical/annotations/instances_val.json \
  --imgs "{IMGS}" \
  --out  runs/diag/medical \
  --device cuda:0 {limit_arg} \
  --models "yoloworld:yolov8s-world.pt,owlv2:google/owlv2-base-patch16-ensemble,groundingdino:IDEA-Research/grounding-dino-tiny" \
  --sam_weights mobile_sam.pt

In [ ]:
import pandas as pd
print(pd.read_csv('runs/diag/medical/fingerprints.csv').to_string(index=False))

## Validation — synthetic controls
**Caveat:** medical AP$\approx$0, so the S (vocab) and temperature controls are partly
degenerate here (a ratio/ranking on near-zero AP). The **blur** control is still meaningful
(SAM localizability). The controls are primarily demonstrated on aerial, where all axes carry
signal; we include them here for completeness and to confirm the L axis behaves in medical too.

In [ ]:
# Temperature control (post-hoc). On medical, watch C_ece; AP is ~0 by construction.
!python tools/synthetic_controls.py --control temperature \
  --ann data/medical/annotations/instances_val.json \
  --results runs/diag/medical/owlv2/results_global.json \
  --out runs/controls/medical_owlv2_temperature.csv

In [ ]:
# Blur control: L should stay high / rise (medical is already localization-bound).
!python tools/synthetic_controls.py --control blur \
  --ann data/medical/annotations/instances_val.json --imgs "{IMGS}" \
  --sam_weights mobile_sam.pt --out runs/controls/medical_blur.csv --limit 200 --device cuda:0

In [ ]:
import pandas as pd, os
for name in ['medical_owlv2_temperature', 'medical_blur']:
    p = f'runs/controls/{name}.csv'
    if os.path.exists(p):
        print(f'== {name} =='); print(pd.read_csv(p).to_string(index=False), '\n')

**Cross-domain read:** compare `runs/diag/medical/fingerprints.csv` to the aerial one. Aerial
S-bound + medical L-bound = the paper's core divergence. Two domains × 3 models = the fingerprint
table; the synthetic controls (aerial) validate the axes.